# 13 — Kapsamlı Ablasyon: No-FE | OHE | XGBoost + LightGBM

**Amaç:** Feature engineering uygulamadan ham veriyi OneHotEncoder ile encode edip XGBoost ve LightGBM ile eğitim.  
Her özellik grubunu sırayla **çıkararak** (drop ablation) hangi grubun F1/AUC/MCC'ye ne kadar katkı yaptığını ölçüyoruz.  
Ek olarak: missing-mask senaryosu ve panel transfer testi.

## Pipeline
- Veri: `YARISMA_TRAIN_MASTER.csv` (2931 satır, 353 sütun)
- Ön işleme: NaN > %50 → drop, kalanlar → mean imputation (train fit), OHE (CAT_+AA_)
- Modeller: XGBoost + LightGBM (paralel), 3-fold CV grid search
- Split: stratified hold-out %20
- Panel testi: CFTR, KANSER, PAH → ablasyon sonrası best-model ile

## Ablasyon Senaryoları (13 senaryo)

| # | Senaryo | Çıkarılan Grup | Açıklama |
|---|---------|----------------|----------|
| 1 | Baseline | — | Tüm özellikler |
| 2 | -CAT_all | CAT_1..CAT_6 | Tüm kategorik (popülasyon + genotip + bölge) |
| 3 | -CAT_pop | CAT_1, CAT_2 | Sadece popülasyon kategorileri |
| 4 | -CAT_geno | CAT_3, CAT_4, CAT_5 | Sadece genotip kategorileri |
| 5 | -CAT_dup | CAT_3, CAT_5 | Özdeş çift (CLAUDE.md uyarısı) |
| 6 | -AA | AA_1, AA_2 | Amino asit kategorileri |
| 7 | -EK | EK_1..EK_9 | Ek sayısal skorlar |
| 8 | -AL_safe | AL_safe grubu | Güvenli AL sütunları (risk taşımayan) |
| 9 | -AL_high_miss | AL_1..6 + AL_27..38 | %80+ eksik bloklar |
| 10 | -AL_miss_leak | AL_16..25 | Eksiklik-etiket korelasyonu yüksek |
| 11 | -AL_low | AL_ (NaN≤%40) | Düşük boşluklu AL |
| 12 | -AL_high | AL_ (NaN>%40) | Yüksek boşluklu AL |
| 13 | -AL_all | Tüm AL_ | Tüm AL özellikleri |

**Missing-mask senaryoları:** Baseline_nomask / +Mask_all / +Mask_highrisk

## Çıktılar
- `results/ablation_ohe_xgb/ablation_results.csv` — 13×2 model tam metrik tablosu
- `results/ablation_ohe_xgb/mask_results.csv` — missing-mask sonuçları
- `results/ablation_ohe_xgb/panel_results.csv` — Panel transfer sonuçları
- `results/ablation_ohe_xgb/fig1..fig5.png` — 5 görselleştirme
- `reports/ablation_ohe_xgb_report.pdf` — Otomatik PDF

In [1]:
# Cell 1: Imports & Config
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from itertools import product as _product

from config import SEED, PROJECT_ROOT, REPORTS_DIR, TEST_SIZE
from src.metrics import compute_all_metrics, optimize_threshold
from src.columns_real import (
    AL_COLS, CAT_COLS, EK_COLS, AA_COLS,
    AL_SAFE_COLS, AL_HIGH_MISSING_COLS, AL_MISSINGNESS_LEAKAGE_RISK,
    CAT_POPULATION_COLS, CAT_GENOTYPE_COLS, CAT_REGION_COLS,
    SUSPECTED_DUPLICATE_PAIRS, get_missing_mask_col_name,
)

# ── Paths ──────────────────────────────────────────────────────────────────────
MASTER_PATH  = os.path.join(PROJECT_ROOT, 'data', 'real_data', 'YARISMA_TRAIN_MASTER.csv')
CFTR_PATH    = os.path.join(PROJECT_ROOT, 'data', 'real_data', 'YARISMA_TRAIN_CFTR.csv')
KANSER_PATH  = os.path.join(PROJECT_ROOT, 'data', 'real_data', 'YARISMA_TRAIN_KANSER.csv')
PAH_PATH     = os.path.join(PROJECT_ROOT, 'data', 'real_data', 'YARISMA_TRAIN_PAH.csv')
RESULTS_DIR  = os.path.join(PROJECT_ROOT, 'results', 'ablation_ohe_xgb')
NAN_THRESH   = 0.50

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

# ── Grid search parametreleri ──────────────────────────────────────────────────
XGB_GRID = {
    'n_estimators':  [100, 200, 300],
    'max_depth':     [4, 6],
    'learning_rate': [0.05, 0.1],
}
XGB_FIXED = dict(
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    eval_metric='logloss', random_state=SEED, verbosity=0,
    objective='binary:logistic',
)

LGBM_GRID = {
    'n_estimators':  [100, 200, 300],
    'num_leaves':    [31, 63],
    'learning_rate': [0.05, 0.1],
}
LGBM_FIXED = dict(
    min_child_samples=20, subsample=0.8, subsample_freq=1,
    random_state=SEED, verbosity=-1, n_jobs=-1,
)

print('Imports OK')
print(f'MASTER : {MASTER_PATH}')
print(f'RESULTS: {RESULTS_DIR}')

Imports OK
MASTER : /Users/tefe/teknofest_model/teknofest_model/data/real_data/YARISMA_TRAIN_MASTER.csv
RESULTS: /Users/tefe/teknofest_model/teknofest_model/results/ablation_ohe_xgb


In [2]:
# Cell 2: Veri Yukleme & Sutun Gruplari
df_raw = pd.read_csv(MASTER_PATH)
print(f'Ham veri: {df_raw.shape}')
print(f'Label dagilimi: {df_raw["Label"].value_counts().to_dict()}')

# NaN > %50 sutunlari drop et (Label/Variant_ID haric)
nan_pct_raw = df_raw.isnull().mean()
drop_cols = [c for c in df_raw.columns
             if nan_pct_raw[c] > NAN_THRESH and c not in ('Label', 'Variant_ID')]
df = df_raw.drop(columns=drop_cols + ['Variant_ID'])
print(f'\nNaN>{int(NAN_THRESH*100)}% drop: {len(drop_cols)} sutun -> kalan: {df.shape}')

# Kalan sutunlari columns_real gruplarına gore tespit et
nan_pct2 = df.isnull().mean()
AL_AVAIL          = [c for c in AL_COLS                      if c in df.columns]
CAT_AVAIL         = [c for c in CAT_COLS                     if c in df.columns]
EK_AVAIL          = [c for c in EK_COLS                      if c in df.columns]
AA_AVAIL          = [c for c in AA_COLS                      if c in df.columns]
AL_SAFE_AVAIL     = [c for c in AL_SAFE_COLS                 if c in df.columns]
AL_HIGHMISS_AVAIL = [c for c in AL_HIGH_MISSING_COLS         if c in df.columns]
AL_MISSLEAK_AVAIL = [c for c in AL_MISSINGNESS_LEAKAGE_RISK  if c in df.columns]
AL_LOW_AVAIL      = [c for c in AL_AVAIL if nan_pct2[c] <= 0.40]
AL_HIGH_AVAIL     = [c for c in AL_AVAIL if nan_pct2[c] >  0.40]

# Ozdes sutun ciftlerini dogrula
dup_pairs_found = [(c1, c2) for c1, c2 in SUSPECTED_DUPLICATE_PAIRS
                   if c1 in df.columns and c2 in df.columns and df[c1].equals(df[c2])]

print(f'\n=== Sutun gruplari (kalan, NaN-filtered) ===')
print(f'AL_AVAIL         : {len(AL_AVAIL)}')
print(f'  AL_SAFE_AVAIL  : {len(AL_SAFE_AVAIL)}')
print(f'  AL_HIGHMISS    : {len(AL_HIGHMISS_AVAIL)}')
print(f'  AL_MISSLEAK    : {len(AL_MISSLEAK_AVAIL)}')
print(f'  AL_LOW (<=40%) : {len(AL_LOW_AVAIL)}')
print(f'  AL_HIGH (>40%) : {len(AL_HIGH_AVAIL)}')
print(f'CAT_AVAIL        : {len(CAT_AVAIL)}')
print(f'EK_AVAIL         : {len(EK_AVAIL)}')
print(f'AA_AVAIL         : {len(AA_AVAIL)}')
print(f'\nOzdes cift onaylandi: {dup_pairs_found}')

y     = df['Label']
X_raw = df.drop(columns=['Label'])
print(f'\nX_raw: {X_raw.shape}  |  pos={int(y.sum())}, neg={int((y==0).sum())}')

Ham veri: (2931, 353)
Label dagilimi: {1: 2149, 0: 782}

NaN>50% drop: 165 sutun -> kalan: (2931, 187)

=== Sutun gruplari (kalan, NaN-filtered) ===
AL_AVAIL         : 171
  AL_SAFE_AVAIL  : 171
  AL_HIGHMISS    : 0
  AL_MISSLEAK    : 0
  AL_LOW (<=40%) : 11
  AL_HIGH (>40%) : 160
CAT_AVAIL        : 4
EK_AVAIL         : 9
AA_AVAIL         : 2

Ozdes cift onaylandi: [('CAT_3', 'CAT_5')]

X_raw: (2931, 186)  |  pos=2149, neg=782


In [3]:
# Cell 3: Train/Test Split + Panel Yukleme
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
print(f'Train: {X_train_raw.shape}  Test: {X_test_raw.shape}')
print(f'Train label: {y_train.value_counts().to_dict()}')
print(f'Test  label: {y_test.value_counts().to_dict()}')


def load_panel(path, master_cols):
    """Panel CSV'sini yukle; MASTER sutun setine gore hizala."""
    p = pd.read_csv(path)
    p = p.drop(columns=['Variant_ID'], errors='ignore')
    y_p = p['Label']
    X_p = p.drop(columns=['Label'])
    for c in master_cols:
        if c not in X_p.columns:
            X_p[c] = np.nan
    X_p = X_p[[c for c in master_cols if c in X_p.columns]]
    return X_p, y_p


panels = {}
for name, path in [('CFTR', CFTR_PATH), ('KANSER', KANSER_PATH), ('PAH', PAH_PATH)]:
    X_p, y_p = load_panel(path, X_raw.columns.tolist())
    panels[name] = (X_p, y_p)
    print(f'Panel {name}: {X_p.shape}  pos={int(y_p.sum())}, neg={int((y_p==0).sum())}')

Train: (2344, 186)  Test: (587, 186)
Train label: {1: 1719, 0: 625}
Test  label: {1: 430, 0: 157}
Panel CFTR: (111, 186)  pos=90, neg=21
Panel KANSER: (388, 186)  pos=268, neg=120
Panel PAH: (372, 186)  pos=310, neg=62


In [4]:
# Cell 4: Preprocess & Egitim Fonksiyonlari (XGBoost + LightGBM)

def preprocess(X_tr_raw, X_te_raw, ohe_cols, numeric_cols, num_means=None):
    """
    1) Mean imputation: numeric_cols. Fit: train. Apply: train + test.
    2) OHE: ohe_cols. Fit: train. Apply: train + test.
    Returns: X_tr (np.ndarray), X_te (np.ndarray), encoder, col_means
    num_means: onceden hesaplanmis mean (panel testi icin); None ise train'den hesapla.
    """
    X_tr, X_te = X_tr_raw.copy(), X_te_raw.copy()

    if numeric_cols:
        col_means = X_tr[numeric_cols].mean() if num_means is None else num_means.reindex(numeric_cols)
        X_tr[numeric_cols] = X_tr[numeric_cols].fillna(col_means)
        X_te[numeric_cols] = X_te[numeric_cols].fillna(col_means)
    else:
        col_means = None

    if ohe_cols:
        for c in ohe_cols:
            X_tr[c] = X_tr[c].fillna('MISSING').astype(str)
            X_te[c] = X_te[c].fillna('MISSING').astype(str)
        enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        enc.fit(X_tr[ohe_cols])
        tr_ohe = enc.transform(X_tr[ohe_cols])
        te_ohe = enc.transform(X_te[ohe_cols])
        tr_num = X_tr[numeric_cols].values if numeric_cols else np.zeros((len(X_tr), 0))
        te_num = X_te[numeric_cols].values if numeric_cols else np.zeros((len(X_te), 0))
        return np.hstack([tr_num, tr_ohe]), np.hstack([te_num, te_ohe]), enc, col_means

    tr_num = X_tr[numeric_cols].values if numeric_cols else np.zeros((len(X_tr), 0))
    te_num = X_te[numeric_cols].values if numeric_cols else np.zeros((len(X_te), 0))
    return tr_num, te_num, None, col_means


def _cv_grid(model_cls, grid, fixed, X_tr, y_tr):
    """3-fold CV grid search. Returns best_combo, best_cv_f1."""
    scale_pos = float((y_tr == 0).sum()) / max(float((y_tr == 1).sum()), 1)
    best_cv_f1, best_combo = -1, None
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    for vals in _product(*grid.values()):
        combo  = dict(zip(grid.keys(), vals))
        params = {**fixed, **combo}
        if model_cls == xgb.XGBClassifier:
            params['scale_pos_weight'] = scale_pos
        else:
            params['class_weight'] = 'balanced'
        fold_f1s = []
        for tr_i, val_i in skf.split(X_tr, y_tr):
            m = model_cls(**params)
            m.fit(X_tr[tr_i], y_tr.iloc[tr_i])
            prob = m.predict_proba(X_tr[val_i])[:, 1]
            _, f1v = optimize_threshold(y_tr.iloc[val_i], prob)
            fold_f1s.append(f1v)
        if np.mean(fold_f1s) > best_cv_f1:
            best_cv_f1 = float(np.mean(fold_f1s))
            best_combo = combo
    return best_combo, best_cv_f1


def train_eval(model_cls, grid, fixed, X_tr, y_tr, X_te, y_te):
    """
    Grid search + final train + threshold-optimized eval.
    Returns: (cv_f1, metrics_dict, best_combo, trained_model, threshold)
    """
    scale_pos = float((y_tr == 0).sum()) / max(float((y_tr == 1).sum()), 1)
    best_combo, best_cv_f1 = _cv_grid(model_cls, grid, fixed, X_tr, y_tr)
    params = {**fixed, **best_combo}
    if model_cls == xgb.XGBClassifier:
        params['scale_pos_weight'] = scale_pos
    else:
        params['class_weight'] = 'balanced'
    model   = model_cls(**params)
    model.fit(X_tr, y_tr)
    y_prob  = model.predict_proba(X_te)[:, 1]
    thr, _  = optimize_threshold(y_te, y_prob)
    y_pred  = (y_prob >= thr).astype(int)
    metrics = compute_all_metrics(y_te, y_pred, y_prob)
    return best_cv_f1, metrics, best_combo, model, thr


print('Fonksiyonlar tanimlandi')

Fonksiyonlar tanimlandi


In [5]:
# Cell 5: Ablasyon Dongusu (13 senaryo x 2 model)

# CAT_3/CAT_5 ozdes cift — dogrulananlar, yoksa fallback
CAT_DUP_COLS = [c for c1, c2 in dup_pairs_found for c in [c1, c2]]
if not CAT_DUP_COLS:
    CAT_DUP_COLS = ['CAT_3', 'CAT_5']

ALL_OHE = [c for c in CAT_AVAIL + AA_AVAIL if c in X_raw.columns]
ALL_NUM = [c for c in AL_AVAIL  + EK_AVAIL if c in X_raw.columns]

scenarios = [
    ('Baseline',      []),
    ('-CAT_all',      CAT_AVAIL),
    ('-CAT_pop',      [c for c in CAT_POPULATION_COLS if c in CAT_AVAIL]),
    ('-CAT_geno',     [c for c in CAT_GENOTYPE_COLS   if c in CAT_AVAIL]),
    ('-CAT_dup',      CAT_DUP_COLS),
    ('-AA',           AA_AVAIL),
    ('-EK',           EK_AVAIL),
    ('-AL_safe',      AL_SAFE_AVAIL),
    ('-AL_high_miss', AL_HIGHMISS_AVAIL),
    ('-AL_miss_leak', AL_MISSLEAK_AVAIL),
    ('-AL_low',       AL_LOW_AVAIL),
    ('-AL_high',      AL_HIGH_AVAIL),
    ('-AL_all',       AL_AVAIL),
]

models_cfg = [
    ('XGBoost',  xgb.XGBClassifier,  XGB_GRID,  XGB_FIXED),
    ('LightGBM', lgb.LGBMClassifier, LGBM_GRID, LGBM_FIXED),
]

results       = []
baseline_info = {}  # {model_name: {model, encoder, col_means, ohe_cols, num_cols, threshold}}

for sc_name, drop_group in scenarios:
    drop_set = set(drop_group)
    cur_ohe  = [c for c in ALL_OHE if c not in drop_set]
    cur_num  = [c for c in ALL_NUM if c not in drop_set]
    sel_cols = cur_ohe + cur_num

    if not sel_cols:
        print(f'[{sc_name}] hic ozellik kalmadi, atlaniyor')
        continue

    X_tr_p, X_te_p, enc, col_means = preprocess(
        X_train_raw[sel_cols], X_test_raw[sel_cols], cur_ohe, cur_num
    )
    print(f'\n[{sc_name}]  X_shape={X_tr_p.shape}  n_drop={len(drop_group)}')

    for m_name, m_cls, m_grid, m_fixed in models_cfg:
        cv_f1, metrics, combo, model, thr = train_eval(
            m_cls, m_grid, m_fixed, X_tr_p, y_train, X_te_p, y_test
        )
        results.append({
            'Senaryo':    sc_name,
            'Model':      m_name,
            'n_features': X_tr_p.shape[1],
            'n_drop':     len(drop_group),
            'cv_f1':      round(cv_f1, 4),
            'f1':         round(metrics['f1'], 4),
            'auc_roc':    round(metrics['auc_roc'], 4),
            'auc_pr':     round(metrics['auc_pr'], 4),
            'precision':  round(metrics['precision'], 4),
            'recall':     round(metrics['recall'], 4),
            'mcc':        round(metrics['mcc'], 4),
            'specificity':round(metrics['specificity'], 4),
            'threshold':  round(thr, 3),
            'best_combo': str(combo),
        })
        print(f'  {m_name:8s}: CV F1={cv_f1:.4f}  HO F1={metrics["f1"]:.4f}  '
              f'AUC={metrics["auc_roc"]:.4f}  MCC={metrics["mcc"]:.4f}')

        if sc_name == 'Baseline':
            baseline_info[m_name] = {
                'model': model, 'encoder': enc, 'col_means': col_means,
                'ohe_cols': cur_ohe, 'num_cols': cur_num, 'threshold': thr
            }

results_df = pd.DataFrame(results)
# Senaryo sirasi icin kategori tipini koru
sc_order = ['Baseline'] + [s for s in results_df['Senaryo'].unique() if s != 'Baseline']
print('\n=== ABLASYON SONUCLARI (F1 pivot) ===')
print(results_df.pivot_table(index='Senaryo', columns='Model', values='f1', aggfunc='first').to_string())


[Baseline]  X_shape=(2344, 280)  n_drop=0
  XGBoost : CV F1=0.8876  HO F1=0.8985  AUC=0.8474  MCC=0.5600
  LightGBM: CV F1=0.8859  HO F1=0.8973  AUC=0.8344  MCC=0.5547

[-CAT_all]  X_shape=(2344, 231)  n_drop=4
  XGBoost : CV F1=0.8838  HO F1=0.8997  AUC=0.8438  MCC=0.5654
  LightGBM: CV F1=0.8847  HO F1=0.8950  AUC=0.8293  MCC=0.5510

[-CAT_pop]  X_shape=(2344, 249)  n_drop=1
  XGBoost : CV F1=0.8853  HO F1=0.9025  AUC=0.8387  MCC=0.5865
  LightGBM: CV F1=0.8843  HO F1=0.9048  AUC=0.8379  MCC=0.5916

[-CAT_geno]  X_shape=(2344, 262)  n_drop=3
  XGBoost : CV F1=0.8883  HO F1=0.8978  AUC=0.8441  MCC=0.5602
  LightGBM: CV F1=0.8860  HO F1=0.8975  AUC=0.8340  MCC=0.5679

[-CAT_dup]  X_shape=(2344, 268)  n_drop=2
  XGBoost : CV F1=0.8902  HO F1=0.8990  AUC=0.8455  MCC=0.5653
  LightGBM: CV F1=0.8858  HO F1=0.8991  AUC=0.8362  MCC=0.5607

[-AA]  X_shape=(2344, 229)  n_drop=2
  XGBoost : CV F1=0.8854  HO F1=0.8889  AUC=0.8199  MCC=0.5193
  LightGBM: CV F1=0.8863  HO F1=0.8942  AUC=0.8118  M

In [6]:
# Cell 6: Panel Transfer Testi (CFTR / KANSER / PAH)
# Baseline model + encoder, her panele uygulanir (test-only, fit yok)

panel_results = []

for m_name in ['XGBoost', 'LightGBM']:
    info     = baseline_info[m_name]
    model_b  = info['model']
    enc_b    = info['encoder']
    means_b  = info['col_means']
    ohe_cols = info['ohe_cols']
    num_cols = info['num_cols']
    thr_b    = info['threshold']
    sel_cols = ohe_cols + num_cols

    for pname, (X_p_raw, y_p) in panels.items():
        X_p = X_p_raw.copy()
        for c in sel_cols:
            if c not in X_p.columns:
                X_p[c] = np.nan
        X_p = X_p[sel_cols]

        # Imputation — MASTER train mean kullan
        if num_cols and means_b is not None:
            X_p[num_cols] = X_p[num_cols].fillna(means_b.reindex(num_cols))

        # OHE transform (fit MASTER train'de yapildi)
        if ohe_cols and enc_b is not None:
            for c in ohe_cols:
                X_p[c] = X_p[c].fillna('MISSING').astype(str)
            p_ohe   = enc_b.transform(X_p[ohe_cols])
            p_num   = X_p[num_cols].values if num_cols else np.zeros((len(X_p), 0))
            X_p_arr = np.hstack([p_num, p_ohe])
        else:
            X_p_arr = X_p[num_cols].values if num_cols else np.zeros((len(X_p), 0))

        if len(np.unique(y_p)) < 2:
            print(f'[{pname}] tek sinifli, atlaniyor')
            continue

        y_prob = model_b.predict_proba(X_p_arr)[:, 1]
        y_pred = (y_prob >= thr_b).astype(int)
        m      = compute_all_metrics(y_p, y_pred, y_prob)

        panel_results.append({
            'Panel': pname, 'Model': m_name,
            'f1':        round(m['f1'], 4),
            'auc_roc':   round(m['auc_roc'], 4),
            'auc_pr':    round(m['auc_pr'], 4),
            'precision': round(m['precision'], 4),
            'recall':    round(m['recall'], 4),
            'mcc':       round(m['mcc'], 4),
            'threshold': round(thr_b, 3),
        })
        print(f'[{pname}] {m_name:8s}: F1={m["f1"]:.4f}  AUC={m["auc_roc"]:.4f}  '
              f'Recall={m["recall"]:.4f}  MCC={m["mcc"]:.4f}')

panel_df = pd.DataFrame(panel_results)
panel_csv = os.path.join(RESULTS_DIR, 'panel_results.csv')
panel_df.to_csv(panel_csv, index=False)
print(f'\nPanel CSV: {panel_csv}')
print(panel_df.pivot_table(index='Panel', columns='Model', values='f1').to_string())

[CFTR] XGBoost : F1=0.9348  AUC=0.9492  Recall=0.9556  MCC=0.6249
[KANSER] XGBoost : F1=0.8930  AUC=0.9019  Recall=0.9813  MCC=0.6090
[PAH] XGBoost : F1=0.9309  AUC=0.7852  Recall=0.9774  MCC=0.4915
[CFTR] LightGBM: F1=0.9457  AUC=0.9386  Recall=0.9667  MCC=0.6888
[KANSER] LightGBM: F1=0.8956  AUC=0.9122  Recall=0.9925  MCC=0.6213
[PAH] LightGBM: F1=0.9344  AUC=0.8406  Recall=0.9871  MCC=0.5143

Panel CSV: /Users/tefe/teknofest_model/teknofest_model/results/ablation_ohe_xgb/panel_results.csv
Model   LightGBM  XGBoost
Panel                    
CFTR      0.9457   0.9348
KANSER    0.8956   0.8930
PAH       0.9344   0.9309


In [7]:
# Cell 7: Missing-Mask Ablasyonu (is_missing_* bayragi)
# CLAUDE.md 'en kritik kontrol': eksiklik bayraklarinin katkisi olculuyor.
# 3 senaryo: Baseline_nomask / +Mask_all / +Mask_highrisk

def add_missing_masks(X_df, mask_cols):
    X_m = X_df.copy()
    for c in mask_cols:
        if c in X_m.columns:
            X_m[get_missing_mask_col_name(c)] = X_m[c].isna().astype(int)
    return X_m


MASK_ALL_COLS      = [c for c in AL_AVAIL + EK_AVAIL if c in X_raw.columns]
MASK_HIGHRISK_COLS = [c for c in AL_HIGHMISS_AVAIL + AL_MISSLEAK_AVAIL if c in X_raw.columns]

mask_scenarios = [
    ('Baseline_nomask', []),
    ('+Mask_all',       MASK_ALL_COLS),
    ('+Mask_highrisk',  MASK_HIGHRISK_COLS),
]

mask_results = []
for sc_name, mask_cols in mask_scenarios:
    X_tr_m = add_missing_masks(X_train_raw[ALL_OHE + ALL_NUM], mask_cols)
    X_te_m = add_missing_masks(X_test_raw[ALL_OHE  + ALL_NUM], mask_cols)

    mask_feat_cols = [get_missing_mask_col_name(c) for c in mask_cols
                      if get_missing_mask_col_name(c) in X_tr_m.columns]
    cur_ohe = ALL_OHE
    cur_num = ALL_NUM + mask_feat_cols

    X_tr_p, X_te_p, _, _ = preprocess(X_tr_m, X_te_m, cur_ohe, cur_num)
    print(f'\n[{sc_name}]  X_shape={X_tr_p.shape}  n_mask={len(mask_feat_cols)}')

    for m_name, m_cls, m_grid, m_fixed in models_cfg:
        cv_f1, metrics, combo, _, thr = train_eval(
            m_cls, m_grid, m_fixed, X_tr_p, y_train, X_te_p, y_test
        )
        mask_results.append({
            'Senaryo': sc_name, 'Model': m_name,
            'n_features': X_tr_p.shape[1],
            'n_mask':     len(mask_feat_cols),
            'cv_f1':      round(cv_f1, 4),
            'f1':         round(metrics['f1'], 4),
            'auc_roc':    round(metrics['auc_roc'], 4),
            'mcc':        round(metrics['mcc'], 4),
            'recall':     round(metrics['recall'], 4),
            'precision':  round(metrics['precision'], 4),
        })
        print(f'  {m_name:8s}: CV F1={cv_f1:.4f}  HO F1={metrics["f1"]:.4f}  '
              f'AUC={metrics["auc_roc"]:.4f}  MCC={metrics["mcc"]:.4f}')

mask_df = pd.DataFrame(mask_results)
print('\n=== MISSING-MASK SONUCLARI ===')
print(mask_df[['Senaryo','Model','n_mask','f1','auc_roc','mcc']].to_string(index=False))


[Baseline_nomask]  X_shape=(2344, 280)  n_mask=0
  XGBoost : CV F1=0.8876  HO F1=0.8985  AUC=0.8474  MCC=0.5600
  LightGBM: CV F1=0.8859  HO F1=0.8973  AUC=0.8344  MCC=0.5547

[+Mask_all]  X_shape=(2344, 460)  n_mask=180
  XGBoost : CV F1=0.8911  HO F1=0.8974  AUC=0.8451  MCC=0.5484
  LightGBM: CV F1=0.8850  HO F1=0.8966  AUC=0.8291  MCC=0.5551

[+Mask_highrisk]  X_shape=(2344, 280)  n_mask=0
  XGBoost : CV F1=0.8876  HO F1=0.8985  AUC=0.8474  MCC=0.5600
  LightGBM: CV F1=0.8859  HO F1=0.8973  AUC=0.8344  MCC=0.5547

=== MISSING-MASK SONUCLARI ===
        Senaryo    Model  n_mask     f1  auc_roc    mcc
Baseline_nomask  XGBoost       0 0.8985   0.8474 0.5600
Baseline_nomask LightGBM       0 0.8973   0.8344 0.5547
      +Mask_all  XGBoost     180 0.8974   0.8451 0.5484
      +Mask_all LightGBM     180 0.8966   0.8291 0.5551
 +Mask_highrisk  XGBoost       0 0.8985   0.8474 0.5600
 +Mask_highrisk LightGBM       0 0.8973   0.8344 0.5547


In [8]:
# Cell 8: Delta Hesabi & CSV Kayit

def get_baseline(df, model_name):
    row = df[(df['Senaryo'] == 'Baseline') & (df['Model'] == model_name)]
    return {k: float(row[k].values[0]) for k in ['f1', 'auc_roc', 'auc_pr', 'mcc']}


for m_name in ['XGBoost', 'LightGBM']:
    bl   = get_baseline(results_df, m_name)
    mask = results_df['Model'] == m_name
    results_df.loc[mask, 'f1_drop']     = bl['f1']      - results_df.loc[mask, 'f1']
    results_df.loc[mask, 'auc_drop']    = bl['auc_roc'] - results_df.loc[mask, 'auc_roc']
    results_df.loc[mask, 'mcc_drop']    = bl['mcc']     - results_df.loc[mask, 'mcc']
    results_df.loc[mask, 'auc_pr_drop'] = bl['auc_pr']  - results_df.loc[mask, 'auc_pr']

for col in ['f1_drop', 'auc_drop', 'mcc_drop', 'auc_pr_drop']:
    results_df[col] = results_df[col].round(4)

csv_path = os.path.join(RESULTS_DIR, 'ablation_results.csv')
results_df.to_csv(csv_path, index=False)
print(f'Ablasyon CSV: {csv_path}')

print('\n=== F1 DROP (Baseline - Senaryo) ===')
pivot_drop = results_df.pivot_table(
    index='Senaryo', columns='Model', values='f1_drop', aggfunc='first'
)
pivot_drop = pivot_drop.reindex([s for s in sc_order if s in pivot_drop.index])
print(pivot_drop.to_string(float_format='{:+.4f}'.format))

mask_csv = os.path.join(RESULTS_DIR, 'mask_results.csv')
mask_df.to_csv(mask_csv, index=False)
print(f'\nMask CSV: {mask_csv}')

Ablasyon CSV: /Users/tefe/teknofest_model/teknofest_model/results/ablation_ohe_xgb/ablation_results.csv

=== F1 DROP (Baseline - Senaryo) ===
Model          LightGBM  XGBoost
Senaryo                         
Baseline        +0.0000  +0.0000
-CAT_all        +0.0023  -0.0012
-CAT_pop        -0.0075  -0.0040
-CAT_geno       -0.0002  +0.0007
-CAT_dup        -0.0018  -0.0005
-AA             +0.0031  +0.0096
-EK             +0.0148  +0.0140
-AL_safe        +0.0210  +0.0275
-AL_high_miss   +0.0000  +0.0000
-AL_miss_leak   +0.0000  +0.0000
-AL_low         +0.0065  +0.0084
-AL_high        +0.0140  +0.0140
-AL_all         +0.0210  +0.0275

Mask CSV: /Users/tefe/teknofest_model/teknofest_model/results/ablation_ohe_xgb/mask_results.csv


In [9]:
# Cell 9: Gorsellestirme (5 grafik)

fig_paths = []
COLORS = {'XGBoost': '#1565C0', 'LightGBM': '#2E7D32'}
scs = [s for s in sc_order if s != 'Baseline']
x   = np.arange(len(scs))
w   = 0.38
ablation_plot = results_df[results_df['Senaryo'] != 'Baseline'].copy()


def get_val(df, sc, model, col):
    row = df[(df['Senaryo'] == sc) & (df['Model'] == model)]
    return float(row[col].values[0]) if len(row) else 0.0


# ── Fig 1: F1 Drop — XGBoost vs LightGBM ──────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))
fig.suptitle('NB13 — F1 Drop: XGBoost vs LightGBM', fontsize=12, fontweight='bold')
for i, (m_name, color) in enumerate(COLORS.items()):
    vals = [get_val(ablation_plot, s, m_name, 'f1_drop') for s in scs]
    bars = ax.bar(x + i * w - w/2, vals, w, label=m_name, color=color, alpha=0.8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, max(bar.get_height(), 0) + 0.0005,
                f'{v:+.4f}', ha='center', va='bottom', fontsize=7, rotation=70)
ax.axhline(0,    color='black', linewidth=0.8, linestyle='--')
ax.axhline(0.03, color='#C62828', linewidth=0.6, linestyle=':', alpha=0.6, label='Kritik esik (0.03)')
ax.set_xticks(x); ax.set_xticklabels(scs, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('F1 Dusus (Baseline - Senaryo)'); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fp = os.path.join(RESULTS_DIR, 'fig1_f1drop_comparison.png')
plt.savefig(fp, bbox_inches='tight', dpi=120); fig_paths.append(fp); plt.close()

# ── Fig 2: 3-Metrik (F1 / AUC-ROC / MCC) — XGBoost ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('NB13 — Metrik Karsilastirmasi (XGBoost)', fontsize=12, fontweight='bold')
xgb_df  = results_df[results_df['Model'] == 'XGBoost'].copy()
sc_lbls = xgb_df['Senaryo'].tolist()
xi = np.arange(len(sc_lbls))
for ax_i, (metric, color, label) in enumerate([
        ('f1',      '#1976D2', 'F1'),
        ('auc_roc', '#388E3C', 'AUC-ROC'),
        ('mcc',     '#E64A19', 'MCC')]):
    axes[ax_i].bar(xi, xgb_df[metric], color=color, alpha=0.85)
    bl_val = float(xgb_df.loc[xgb_df['Senaryo'] == 'Baseline', metric].values[0])
    axes[ax_i].axhline(bl_val, color='navy', linewidth=1.2, linestyle='--', label=f'Baseline={bl_val:.4f}')
    axes[ax_i].set_xticks(xi)
    axes[ax_i].set_xticklabels(sc_lbls, rotation=40, ha='right', fontsize=8)
    axes[ax_i].set_title(label); axes[ax_i].legend(fontsize=8); axes[ax_i].grid(axis='y', alpha=0.3)
plt.tight_layout()
fp = os.path.join(RESULTS_DIR, 'fig2_metrics_xgb.png')
plt.savefig(fp, bbox_inches='tight', dpi=120); fig_paths.append(fp); plt.close()

# ── Fig 3: Panel Transfer Heatmap (F1) ────────────────────────────────────
if not panel_df.empty:
    pt  = panel_df.pivot_table(index='Panel', columns='Model', values='f1', aggfunc='first')
    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(pt.values, cmap='RdYlGn', vmin=0.5, vmax=1.0, aspect='auto')
    ax.set_xticks(range(len(pt.columns))); ax.set_xticklabels(pt.columns)
    ax.set_yticks(range(len(pt.index)));   ax.set_yticklabels(pt.index)
    for r in range(len(pt.index)):
        for c in range(len(pt.columns)):
            ax.text(c, r, f'{pt.values[r, c]:.4f}', ha='center', va='center', fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax, label='F1')
    ax.set_title('Panel Transfer F1 (Baseline model, test-only)')
    plt.tight_layout()
    fp = os.path.join(RESULTS_DIR, 'fig3_panel_heatmap.png')
    plt.savefig(fp, bbox_inches='tight', dpi=120); fig_paths.append(fp); plt.close()

# ── Fig 4: Missing-Mask Etkisi ────────────────────────────────────────────
if not mask_df.empty:
    mask_scs = mask_df['Senaryo'].unique()
    xi2 = np.arange(len(mask_scs))
    fig, ax = plt.subplots(figsize=(8, 4))
    fig.suptitle('Missing-Mask Etkisi (is_missing_* bayragi)', fontsize=11, fontweight='bold')
    for i, (m_name, color) in enumerate(COLORS.items()):
        vals = [get_val(mask_df, s, m_name, 'f1') for s in mask_scs]
        ax.bar(xi2 + i * 0.35 - 0.175, vals, 0.35, label=m_name, color=color, alpha=0.8)
        for j, v in enumerate(vals):
            ax.text(xi2[j] + i * 0.35 - 0.175, v + 0.001, f'{v:.4f}', ha='center', fontsize=9)
    ax.set_xticks(xi2); ax.set_xticklabels(mask_scs, fontsize=9)
    ax.set_ylabel('Hold-out F1'); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    fp = os.path.join(RESULTS_DIR, 'fig4_missing_mask.png')
    plt.savefig(fp, bbox_inches='tight', dpi=120); fig_paths.append(fp); plt.close()

# ── Fig 5: AUC-PR Drop (sinif dengesizligi icin kritik) ───────────────────
fig, ax = plt.subplots(figsize=(13, 5))
fig.suptitle('NB13 — AUC-PR Drop (sinif dengesizligi etkisi)', fontsize=11, fontweight='bold')
for i, (m_name, color) in enumerate(COLORS.items()):
    vals = [get_val(ablation_plot, s, m_name, 'auc_pr_drop') for s in scs]
    ax.bar(x + i * w - w/2, vals, w, label=m_name, color=color, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x); ax.set_xticklabels(scs, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('AUC-PR Dusus'); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fp = os.path.join(RESULTS_DIR, 'fig5_aucpr_drop.png')
plt.savefig(fp, bbox_inches='tight', dpi=120); fig_paths.append(fp); plt.close()

print('Kaydedilen grafikler:')
for fp in fig_paths:
    print(f'  {fp}')

Kaydedilen grafikler:
  /Users/tefe/teknofest_model/teknofest_model/results/ablation_ohe_xgb/fig1_f1drop_comparison.png
  /Users/tefe/teknofest_model/teknofest_model/results/ablation_ohe_xgb/fig2_metrics_xgb.png
  /Users/tefe/teknofest_model/teknofest_model/results/ablation_ohe_xgb/fig3_panel_heatmap.png
  /Users/tefe/teknofest_model/teknofest_model/results/ablation_ohe_xgb/fig4_missing_mask.png
  /Users/tefe/teknofest_model/teknofest_model/results/ablation_ohe_xgb/fig5_aucpr_drop.png


In [10]:
# Cell 10: Ozet & Yorumlar
print('=' * 60)
print('ABLASYON OZETI — NB13 (No-FE | OHE | XGBoost + LightGBM)')
print('=' * 60)

for m_name in ['XGBoost', 'LightGBM']:
    bl   = get_baseline(results_df, m_name)
    df_m = results_df[(results_df['Model'] == m_name) &
                      (results_df['Senaryo'] != 'Baseline')].copy()
    top  = df_m.sort_values('f1_drop', ascending=False).iloc[0]
    bot  = df_m.sort_values('f1_drop', ascending=True).iloc[0]
    print(f'\n--- {m_name} ---')
    print(f'Baseline : F1={bl["f1"]:.4f}  AUC={bl["auc_roc"]:.4f}  MCC={bl["mcc"]:.4f}')
    print(f'En KRITIK grup : {top["Senaryo"]:15s} (F1 dusus: {top["f1_drop"]:+.4f})')
    print(f'En AZ ETKILI   : {bot["Senaryo"]:15s} (F1 degisim: {bot["f1_drop"]:+.4f})')
    high_impact = df_m[df_m['f1_drop'] > 0.03]
    if not high_impact.empty:
        print(f'Kritik gruplar (>0.03 F1 dusus): {high_impact["Senaryo"].tolist()}')

print('\n--- Missing-Mask Etkisi ---')
for m_name in ['XGBoost', 'LightGBM']:
    bl_val = get_val(mask_df, 'Baseline_nomask', m_name, 'f1')
    for sc in ['+Mask_all', '+Mask_highrisk']:
        delta = get_val(mask_df, sc, m_name, 'f1') - bl_val
        print(f'  {m_name:8s} {sc:18s}: F1 degisim={delta:+.4f}')

if not panel_df.empty:
    print('\n--- Panel Transfer (Baseline model) ---')
    print(panel_df[['Panel','Model','f1','auc_roc','recall','mcc']].to_string(index=False))

print('\n--- Senaryolar (XGBoost, F1 dususune gore sirali) ---')
xgb_sorted = results_df[results_df['Model'] == 'XGBoost'].sort_values('f1_drop', ascending=False)
print(xgb_sorted[['Senaryo','n_features','f1','f1_drop','auc_roc','mcc']].to_string(index=False))

ABLASYON OZETI — NB13 (No-FE | OHE | XGBoost + LightGBM)

--- XGBoost ---
Baseline : F1=0.8985  AUC=0.8474  MCC=0.5600
En KRITIK grup : -AL_safe        (F1 dusus: +0.0275)
En AZ ETKILI   : -CAT_pop        (F1 degisim: -0.0040)

--- LightGBM ---
Baseline : F1=0.8973  AUC=0.8344  MCC=0.5547
En KRITIK grup : -AL_safe        (F1 dusus: +0.0210)
En AZ ETKILI   : -CAT_pop        (F1 degisim: -0.0075)

--- Missing-Mask Etkisi ---
  XGBoost  +Mask_all         : F1 degisim=-0.0011
  XGBoost  +Mask_highrisk    : F1 degisim=+0.0000
  LightGBM +Mask_all         : F1 degisim=-0.0007
  LightGBM +Mask_highrisk    : F1 degisim=+0.0000

--- Panel Transfer (Baseline model) ---
 Panel    Model     f1  auc_roc  recall    mcc
  CFTR  XGBoost 0.9348   0.9492  0.9556 0.6249
KANSER  XGBoost 0.8930   0.9019  0.9813 0.6090
   PAH  XGBoost 0.9309   0.7852  0.9774 0.4915
  CFTR LightGBM 0.9457   0.9386  0.9667 0.6888
KANSER LightGBM 0.8956   0.9122  0.9925 0.6213
   PAH LightGBM 0.9344   0.8406  0.9871 0.5143

--

In [13]:
# Cell 11: PDF Rapor (otomatik)
from fpdf import FPDF
from datetime import datetime


class AblationReport(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 9)
        self.cell(0, 7, 'Teknofest - NB13 Kapsamli Ablasyon Raporu (No-FE | OHE | XGB + LGBM)',
                  align='C', new_x='LMARGIN', new_y='NEXT')
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(2)

    def footer(self):
        self.set_y(-14)
        self.set_font('Helvetica', 'I', 8)
        self.cell(0, 8, f'Sayfa {self.page_no()}/{{nb}}', align='C')

    def section(self, title):
        self.set_font('Helvetica', 'B', 11)
        self.ln(3)
        self.cell(0, 7, title, new_x='LMARGIN', new_y='NEXT')
        self.set_font('Helvetica', '', 9)

    def line_item(self, text):
        self.cell(0, 5, text, new_x='LMARGIN', new_y='NEXT')

    def draw_table(self, headers, col_widths, rows, bold_first=False):
        self.set_font('Helvetica', 'B', 7)
        for w, h in zip(col_widths, headers):
            self.cell(w, 6, h, border=1, align='C')
        self.ln()
        self.set_font('Helvetica', '', 7)
        for i, row in enumerate(rows):
            if bold_first and i == 0:
                self.set_font('Helvetica', 'B', 7)
            for w, v in zip(col_widths, row):
                self.cell(w, 5, str(v), border=1, align='C')
            self.ln()
            if bold_first and i == 0:
                self.set_font('Helvetica', '', 7)


pdf = AblationReport()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=18)
pdf.add_page()

# Kapak
pdf.set_font('Helvetica', 'B', 16)
pdf.ln(20)
pdf.cell(0, 12, 'NB13 - Kapsamli Ablasyon Deneyi', align='C', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 11)
pdf.cell(0, 8, 'No Feature Engineering | OHE | XGBoost + LightGBM',
         align='C', new_x='LMARGIN', new_y='NEXT')
pdf.cell(0, 8, f'Tarih: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
         align='C', new_x='LMARGIN', new_y='NEXT')
pdf.ln(6)

# 1. Pipeline
pdf.section('1. Pipeline & Veri')
for ln in [
    f'Veri: YARISMA_TRAIN_MASTER.csv  ({df_raw.shape[0]} satir, {df_raw.shape[1]} sutun)',
    f'NaN>{int(NAN_THRESH*100)}% drop: {len(drop_cols)} sutun cikarildi -> kalan {X_raw.shape[1]}',
    f'Encoding: OHE (CAT_+AA_), mean imputation (AL_+EK_) - fit sadece train',
    f'Model: XGBoost + LightGBM, 3-fold CV grid search (12 combo)',
    f'Split: stratified hold-out %{int(TEST_SIZE*100)} (train={X_train_raw.shape[0]}, test={X_test_raw.shape[0]})',
    f'Senaryo sayisi: {results_df["Senaryo"].nunique()} ablasyon + 3 missing-mask',
]:
    pdf.line_item(ln)

# 2. Ablasyon Tablosu
pdf.section('2. Ablasyon Sonuclari (Hold-out)')
headers1   = ['Senaryo', 'n_feat', 'CV F1', 'F1', 'dF1', 'AUC', 'dAUC', 'AUC-PR', 'MCC', 'Recall']
col_widths1 = [24, 12, 14, 14, 14, 14, 14, 14, 14, 14]
for m_name in ['XGBoost', 'LightGBM']:
    pdf.set_font('Helvetica', 'B', 9)
    pdf.cell(0, 6, f'  Model: {m_name}', new_x='LMARGIN', new_y='NEXT')
    df_m = results_df[results_df['Model'] == m_name]
    rows = [
        [row['Senaryo'], int(row['n_features']),
         f"{row['cv_f1']:.4f}", f"{row['f1']:.4f}", f"{row['f1_drop']:+.4f}",
         f"{row['auc_roc']:.4f}", f"{row['auc_drop']:+.4f}", f"{row['auc_pr']:.4f}",
         f"{row['mcc']:.4f}", f"{row['recall']:.4f}"]
        for _, row in df_m.iterrows()
    ]
    pdf.draw_table(headers1, col_widths1, rows, bold_first=True)
    pdf.ln(2)

# 3. Missing-Mask
pdf.section('3. Missing-Mask Ablasyonu')
headers2   = ['Senaryo', 'Model', 'n_mask', 'F1', 'AUC-ROC', 'MCC', 'Recall']
col_widths2 = [32, 22, 16, 18, 18, 18, 16]
rows2 = [
    [row['Senaryo'], row['Model'], int(row['n_mask']),
     f"{row['f1']:.4f}", f"{row['auc_roc']:.4f}", f"{row['mcc']:.4f}", f"{row['recall']:.4f}"]
    for _, row in mask_df.iterrows()
]
pdf.draw_table(headers2, col_widths2, rows2)

# 4. Panel Transfer
if not panel_df.empty:
    pdf.section('4. Panel Transfer Sonuclari')
    headers3   = ['Panel', 'Model', 'F1', 'AUC-ROC', 'AUC-PR', 'Precision', 'Recall', 'MCC']
    col_widths3 = [18, 18, 18, 18, 18, 18, 18, 16]
    rows3 = [
        [row['Panel'], row['Model'],
         f"{row['f1']:.4f}", f"{row['auc_roc']:.4f}", f"{row['auc_pr']:.4f}",
         f"{row['precision']:.4f}", f"{row['recall']:.4f}", f"{row['mcc']:.4f}"]
        for _, row in panel_df.iterrows()
    ]
    pdf.draw_table(headers3, col_widths3, rows3)

# 5. Bulgular
pdf.section('5. Bulgular')
for m_name in ['XGBoost', 'LightGBM']:
    bl   = get_baseline(results_df, m_name)
    df_m = results_df[(results_df['Model'] == m_name) & (results_df['Senaryo'] != 'Baseline')]
    top  = df_m.sort_values('f1_drop', ascending=False).iloc[0]
    bot  = df_m.sort_values('f1_drop', ascending=True).iloc[0]
    pdf.line_item(f'{m_name} Baseline: F1={bl["f1"]:.4f}  AUC={bl["auc_roc"]:.4f}  MCC={bl["mcc"]:.4f}')
    pdf.line_item(f'  En kritik: {top["Senaryo"]} (F1 dusus: {top["f1_drop"]:+.4f})')
    pdf.line_item(f'  En az etkili: {bot["Senaryo"]} (F1 degisim: {bot["f1_drop"]:+.4f})')
pdf.ln(2)
pdf.set_font('Helvetica', '', 9)
pdf.multi_cell(0, 4,
    'Yorum: Pozitif dF1 = grubun cikarilmasi F1 dusurdu (grup faydali). '
    'Negatif dF1 = grup cikarildiginda model iyilesti (gurultu/leakage riski). '
    'En yuksek dF1 olan grup, feature engineering ile zenginlestirilmeye oncelikli aday. '
    'Missing-mask katkisi pozitifse eksiklik deseni etiketle koreleli (leakage riski).')

# Grafikler
for fp in fig_paths:
    if os.path.exists(fp):
        pdf.add_page()
        title = os.path.basename(fp).replace('.png', '').replace('_', ' ').title()
        pdf.set_font('Helvetica', 'B', 10)
        pdf.cell(0, 8, title, new_x='LMARGIN', new_y='NEXT')
        try:
            pdf.image(fp, x=10, w=190)
        except Exception as e:
            pdf.set_font('Helvetica', '', 8)
            pdf.cell(0, 6, f'Grafik yuklenemedi: {e}', new_x='LMARGIN', new_y='NEXT')

report_path = os.path.join(REPORTS_DIR, 'ablation_ohe_xgb_report.pdf')
pdf.output(report_path)
print(f'PDF rapor: {report_path}')

PDF rapor: /Users/tefe/teknofest_model/teknofest_model/reports/ablation_ohe_xgb_report.pdf
